# PySpark HW


### Importing Libraries

In [0]:
# Cell 1: Import libraries and initialize Spark session
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import time
from pyspark.sql.functions import col, year, month, hour, dayofweek,when,  round as spark_round
from pyspark.sql.functions import count, avg, sum as spark_sum, max as spark_max, min as spark_min

print(f"Spark Version: {spark.version}")

Spark Version: 4.0.0


###  Load Yellow Taxi Data

In [0]:
taxi_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/databricks-datasets/nyctaxi/tripdata/yellow/yellow_tripdata_2019-*.csv.gz")

print("Data loaded successfully!")
taxi_df.show(5, truncate=False)

Data loaded successfully!
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|1       |2019-03-01 00:24:41 |2019-03-01 00:25:31  |1              |0.0          |1         |N                 |145         |145         |2           |2.5        |0.5  |0.5    |0.0       |0.0     

### Load Taxi Zone Lookup Data (for Join)

In [0]:
# Cell 6: Load taxi zone lookup data for join operation
zone_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/databricks-datasets/nyctaxi/taxizone/taxi_zone_lookup.csv")

print("Taxi Zone Lookup data loaded!")
print("\nZone Schema:")
zone_df.printSchema()
print("\nSample zones:")
zone_df.show(10, truncate=False)

Taxi Zone Lookup data loaded!

Zone Schema:
root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)


Sample zones:
+----------+-------------+-----------------------+------------+
|LocationID|Borough      |Zone                   |service_zone|
+----------+-------------+-----------------------+------------+
|1         |EWR          |Newark Airport         |EWR         |
|2         |Queens       |Jamaica Bay            |Boro Zone   |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4         |Manhattan    |Alphabet City          |Yellow Zone |
|5         |Staten Island|Arden Heights          |Boro Zone   |
|6         |Staten Island|Arrochar/Fort Wadsworth|Boro Zone   |
|7         |Queens       |Astoria                |Boro Zone   |
|8         |Queens       |Astoria Park           |Boro Zone   |
|9         |Queens       |Auburndale             |Boro Zone   |
|10  

### Apply filters early in the pipeline for optimization

In [0]:
# Filter 1: Remove invalid trips (fare > 0, distance > 0, passenger count > 0)
# Filter 2: Focus on 2019 trips only with reasonable trip distances

# Apply filters EARLY for performance optimization
filtered_trips = taxi_df \
    .filter(col("fare_amount") > 0) \
    .filter(col("trip_distance") > 0) \
    .filter(col("trip_distance") < 100) \
    .filter(col("passenger_count") > 0) \
    .filter(col("passenger_count") <= 6) \
    .filter(col("total_amount") > 0)

print("Filters applied successfully!")
print("Sample filtered data:")
filtered_trips.show(5)

# Show how many records we're working with (using a limit to avoid RDD count issues)
print("\nFirst few statistics:")
filtered_trips.select("trip_distance", "fare_amount", "total_amount").describe().show()

Filters applied successfully!
Sample filtered data:
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|       1| 2019-03-01 00:25:27|  2019-03-01 00:36:37|              2|          3.7|         1|                 N|          95|         130|           1|       13.0|  0.5| 

### Add new columns using withColumn

In [0]:
# Extract temporal features and calculate new metrics

# Add multiple calculated columns
transformed_trips = filtered_trips \
    .withColumn("pickup_hour", hour(col("tpep_pickup_datetime"))) \
    .withColumn("pickup_day", dayofweek(col("tpep_pickup_datetime"))) \
    .withColumn("pickup_month", month(col("tpep_pickup_datetime"))) \
    .withColumn("fare_per_mile", spark_round(col("fare_amount") / col("trip_distance"), 2)) \
    .withColumn("tip_percentage", spark_round((col("tip_amount") / col("fare_amount")) * 100, 2)) \
    .withColumn("is_weekend", when(col("pickup_day").isin([1, 7]), "Yes").otherwise("No")) \
    .withColumn("time_of_day", 
                when(col("pickup_hour").between(6, 11), "Morning")
                .when(col("pickup_hour").between(12, 17), "Afternoon")
                .when(col("pickup_hour").between(18, 22), "Evening")
                .otherwise("Night"))

print("Column transformations applied!")
print("\nNew columns added: pickup_hour, pickup_day, pickup_month, fare_per_mile, tip_percentage, is_weekend, time_of_day")
print("\nSample transformed data:")
transformed_trips.select("tpep_pickup_datetime", "pickup_hour", "time_of_day", 
                         "trip_distance", "fare_per_mile", "tip_percentage", "is_weekend").show(10)

Column transformations applied!

New columns added: pickup_hour, pickup_day, pickup_month, fare_per_mile, tip_percentage, is_weekend, time_of_day

Sample transformed data:
+--------------------+-----------+-----------+-------------+-------------+--------------+----------+
|tpep_pickup_datetime|pickup_hour|time_of_day|trip_distance|fare_per_mile|tip_percentage|is_weekend|
+--------------------+-----------+-----------+-------------+-------------+--------------+----------+
| 2019-03-01 00:25:27|          0|      Night|          3.7|         3.51|          5.38|        No|
| 2019-03-01 00:05:21|          0|      Night|         14.1|         2.91|         24.63|        No|
| 2019-03-01 00:48:55|          0|      Night|          9.6|         2.81|           0.0|        No|
| 2019-03-01 00:11:42|          0|      Night|          0.8|         6.88|         54.55|        No|
| 2019-03-01 00:45:03|          0|      Night|          1.2|          5.0|           0.0|        No|
| 2019-03-01 00:02:3

### Join taxi trips with zone lookup data

In [0]:
# Join on pickup location to get borough and zone names

# Join trips with zone data for pickup location
trips_with_zones = transformed_trips \
    .join(zone_df, 
          transformed_trips.PULocationID == zone_df.LocationID, 
          "inner") \
    .select(transformed_trips["*"], 
            zone_df["Borough"].alias("pickup_borough"),
            zone_df["Zone"].alias("pickup_zone"))

print("Join operation completed!")
print("\nJoined data with borough and zone information:")
trips_with_zones.select("tpep_pickup_datetime", "pickup_zone", "pickup_borough", 
                        "trip_distance", "fare_amount", "time_of_day").show(10, truncate=False)

# Show distinct boroughs
print("\nDistinct Boroughs:")
trips_with_zones.select("pickup_borough").distinct().show()

Join operation completed!

Joined data with borough and zone information:
+--------------------+-----------------------------+--------------+-------------+-----------+-----------+
|tpep_pickup_datetime|pickup_zone                  |pickup_borough|trip_distance|fare_amount|time_of_day|
+--------------------+-----------------------------+--------------+-------------+-----------+-----------+
|2019-03-01 00:25:27 |Forest Hills                 |Queens        |3.7          |13.0       |Night      |
|2019-03-01 00:05:21 |West Village                 |Manhattan     |14.1         |41.0       |Night      |
|2019-03-01 00:48:55 |LaGuardia Airport            |Queens        |9.6          |27.0       |Night      |
|2019-03-01 00:11:42 |Clinton East                 |Manhattan     |0.8          |5.5        |Night      |
|2019-03-01 00:45:03 |West Chelsea/Hudson Yards    |Manhattan     |1.2          |6.0        |Night      |
|2019-03-01 00:02:37 |Upper West Side South        |Manhattan     |0.6        

### GroupBy with multiple aggregations

In [0]:
# Aggregation 1: By Borough and Time of Day
borough_analysis = trips_with_zones \
    .groupBy("pickup_borough", "time_of_day") \
    .agg(
        count("*").alias("total_trips"),
        avg("trip_distance").alias("avg_distance"),
        avg("fare_amount").alias("avg_fare"),
        avg("tip_percentage").alias("avg_tip_pct"),
        spark_sum("total_amount").alias("total_revenue")
    ) \
    .orderBy("pickup_borough", "time_of_day")

print("Borough Analysis (by time of day):")
borough_analysis.show(20, truncate=False)

# Aggregation 2: By Hour and Weekend
hourly_analysis = trips_with_zones \
    .groupBy("pickup_hour", "is_weekend") \
    .agg(
        count("*").alias("total_trips"),
        avg("fare_per_mile").alias("avg_fare_per_mile"),
        spark_max("trip_distance").alias("max_distance"),
        spark_min("fare_amount").alias("min_fare")
    ) \
    .orderBy("pickup_hour")

print("\n\nHourly Analysis (Weekend vs Weekday):")
hourly_analysis.show(24, truncate=False)

Borough Analysis (by time of day):
+--------------+-----------+-----------+------------------+------------------+------------------+--------------------+
|pickup_borough|time_of_day|total_trips|avg_distance      |avg_fare          |avg_tip_pct       |total_revenue       |
+--------------+-----------+-----------+------------------+------------------+------------------+--------------------+
|Bronx         |Afternoon  |33322      |5.7289928575715745|21.77996608847008 |2.7412640297701225|817101.560000023    |
|Bronx         |Evening    |20489      |5.407909121967887 |19.742011811215768|3.6416882229489   |466454.86000000144  |
|Bronx         |Morning    |41482      |6.793278530446944 |23.87639241116628 |1.7581806566703633|1100395.3099999991  |
|Bronx         |Night      |17272      |5.175823876794815 |18.408778948587308|4.519309865678555 |366110.8399999972   |
|Brooklyn      |Afternoon  |202301     |4.232254264684799 |17.582304091428124|9.555417224828393 |4243872.3199991565  |
|Brooklyn    

### Create temp views and run SQL queries

In [0]:

# Register DataFrames as temporary SQL tables

trips_with_zones.createOrReplaceTempView("taxi_trips")
zone_df.createOrReplaceTempView("zones")

print("Temp views created: taxi_trips, zones")

# SQL Query 1: Top 10 most popular pickup zones by revenue
print("\n SQL Query 1: Top 10 Zones by Total Revenue")
top_zones_revenue = spark.sql("""
    SELECT 
        pickup_zone,
        pickup_borough,
        COUNT(*) as trip_count,
        ROUND(AVG(fare_amount), 2) as avg_fare,
        ROUND(SUM(total_amount), 2) as total_revenue
    FROM taxi_trips
    GROUP BY pickup_zone, pickup_borough
    ORDER BY total_revenue DESC
    LIMIT 10
""")
top_zones_revenue.show(10, truncate=False)

# SQL Query 2: Average metrics by borough and weekend
print("\nSQL Query 2: Borough Performance (Weekend vs Weekday)")
borough_weekend_analysis = spark.sql("""
    SELECT 
        pickup_borough,
        is_weekend,
        COUNT(*) as total_trips,
        ROUND(AVG(trip_distance), 2) as avg_distance,
        ROUND(AVG(fare_amount), 2) as avg_fare,
        ROUND(AVG(tip_percentage), 2) as avg_tip_pct
    FROM taxi_trips
    WHERE pickup_borough != 'Unknown'
    GROUP BY pickup_borough, is_weekend
    ORDER BY pickup_borough, is_weekend
""")
borough_weekend_analysis.show(truncate=False)

Temp views created: taxi_trips, zones

 SQL Query 1: Top 10 Zones by Total Revenue
+----------------------------+--------------+----------+--------+--------------+
|pickup_zone                 |pickup_borough|trip_count|avg_fare|total_revenue |
+----------------------------+--------------+----------+--------+--------------+
|JFK Airport                 |Queens        |2611642   |45.25   |1.4740370017E8|
|LaGuardia Airport           |Queens        |2103399   |31.53   |9.150573788E7 |
|Midtown Center              |Manhattan     |3367343   |11.97   |5.996693828E7 |
|Times Sq/Theatre District   |Manhattan     |2823306   |13.07   |5.387480767E7 |
|Upper East Side South       |Manhattan     |3558439   |9.65    |5.261943532E7 |
|Penn Station/Madison Sq West|Manhattan     |2942733   |12.23   |5.24234168E7  |
|Midtown East                |Manhattan     |2973828   |11.58   |5.183843984E7 |
|Upper East Side North       |Manhattan     |3218284   |9.73    |4.796683616E7 |
|Clinton East             

### Execution plan for optimization analysis

In [0]:
# This shows how Spark optimizes the query
print(" Execution Plan for Borough Analysis ")
print("\nThis shows:")
print("- Filter pushdown optimization")
print("- Join strategy used")
print("- Aggregation execution")
print("\n")

# Show the physical plan
borough_analysis.explain(mode="formatted")

print("\n\n Execution Plan for SQL Query (Top Zones by Revenue) ")
top_zones_revenue.explain(mode="formatted")

=== Execution Plan for Borough Analysis ===

This shows:
- Filter pushdown optimization
- Join strategy used
- Aggregation execution


== Physical Plan ==
AdaptiveSparkPlan (25)
+- == Initial Plan ==
   ColumnarToRow (24)
   +- PhotonResultStage (23)
      +- PhotonSort (22)
         +- PhotonShuffleExchangeSource (21)
            +- PhotonShuffleMapStage (20)
               +- PhotonShuffleExchangeSink (19)
                  +- PhotonGroupingAgg (18)
                     +- PhotonShuffleExchangeSource (17)
                        +- PhotonShuffleMapStage (16)
                           +- PhotonShuffleExchangeSink (15)
                              +- PhotonGroupingAgg (14)
                                 +- PhotonProject (13)
                                    +- PhotonBroadcastHashJoin Inner (12)
                                       :- PhotonProject (5)
                                       :  +- PhotonProject (4)
                                       :     +- PhotonFilter (3)

### Caching Optimization (Bonus)

In [0]:
# First, let's run an aggregation WITHOUT caching
print(" Performance Test: WITHOUT Caching ")
start_time = time.time()
result1 = trips_with_zones.groupBy("pickup_borough").agg(count("*").alias("trips")).show()
time1 = time.time() - start_time
print(f"Time WITHOUT cache (1st run): {time1:.2f} seconds")

start_time = time.time()
result2 = trips_with_zones.groupBy("pickup_borough").agg(avg("fare_amount").alias("avg_fare")).show()
time2 = time.time() - start_time
print(f"Time WITHOUT cache (2nd run): {time2:.2f} seconds")

# Now WITH caching
print("\n Performance Test: WITH Caching ")
trips_with_zones.cache()

start_time = time.time()
result3 = trips_with_zones.groupBy("pickup_borough").agg(count("*").alias("trips")).show()
time3 = time.time() - start_time
print(f"Time WITH cache (1st run): {time3:.2f} seconds")

start_time = time.time()
result4 = trips_with_zones.groupBy("pickup_borough").agg(avg("fare_amount").alias("avg_fare")).show()
time4 = time.time() - start_time
print(f"Time WITH cache (2nd run): {time4:.2f} seconds")

print(f"\n Performance Improvement ")
print(f"2nd query speedup: {time2/time4:.2f}x faster with cache")

# Unpersist to free memory
trips_with_zones.unpersist()

=== Performance Test: WITHOUT Caching ===
+--------------+--------+
|pickup_borough|   trips|
+--------------+--------+
|      Brooklyn|  881670|
|         Bronx|  112565|
|     Manhattan|74513046|
|        Queens| 5523985|
|       Unknown|  712472|
| Staten Island|    2768|
|           EWR|    2269|
+--------------+--------+

Time WITHOUT cache (1st run): 26.75 seconds
+--------------+------------------+
|pickup_borough|          avg_fare|
+--------------+------------------+
|      Brooklyn|16.239630258486734|
|         Bronx| 21.66430995424866|
|     Manhattan| 11.37370696844147|
|        Queens|  36.6876016698089|
|       Unknown|14.526054020368527|
| Staten Island| 51.12746748554914|
|           EWR|  81.0821859850154|
+--------------+------------------+

Time WITHOUT cache (2nd run): 26.05 seconds

=== Performance Test: WITH Caching ===


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6925214654411519>, line 15
     13 # Now WITH caching
     14 print("\n=== Performance Test: WITH Caching ===")
---> 15 trips_with_zones.cache()
     17 start_time = time.time()
     18 result3 = trips_with_zones.groupBy("pickup_borough").agg(count("*").alias("trips")).show()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2093, in DataFrame.cache(self)
   2092 def cache(self) -> ParentDataFrame:
-> 2093     return self.persist()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2100, in DataFrame.persist(self, storageLevel)
   2095 def persist(
   2096     self,
   2097     storageLevel: StorageLevel = (StorageLevel.MEMORY_AND_DISK_DESER),
   2098 ) -> ParentDataFrame:
   2099     relation = self._plan.plan(self._session.client)
-> 2100     self._

## Actions vs Transformations

In [0]:
# Transformations (Lazy) vs Actions (Eager)

print("TRANSFORMATIONS (Lazy Evaluation")
print("Transformations don't execute immediately - they build a execution plan\n")

# Apply multiple transformations
print("Applying transformations...")
lazy_df = taxi_df \
    .filter(col("fare_amount") > 10) \
    .withColumn("high_fare", col("fare_amount") > 50) \
    .select("fare_amount", "trip_distance", "high_fare")

print("Transformations applied (no computation happened yet!)")
print("No data was actually processed - Spark just recorded the plan\n")

print("ACTIONS (Eager Evaluation)")
print("Actions trigger actual computation\n")

# Action 1: show() - triggers computation
print("Action 1: show() - This EXECUTES the transformations")
import time
start = time.time()
lazy_df.show(5)
print(f"Execution time: {time.time() - start:.2f} seconds\n")

# Action 2: count() equivalent using groupBy
print("Action 2: Display aggregation - Another action that triggers execution")
start = time.time()
lazy_df.groupBy().agg(count("*").alias("total")).show()
print(f"Execution time: {time.time() - start:.2f} seconds\n")

# Action 3: write - triggers computation
print("Action 3: collect() equivalent - Collecting sample data")
start = time.time()
sample_data = lazy_df.limit(10).collect()
print(f"Collected {len(sample_data)} rows")
print(f"Execution time: {time.time() - start:.2f} seconds")

print("\n KEY TAKEAWAYS")
print("• Transformations (filter, select, withColumn, join, groupBy) = LAZY")
print("• Actions (show, count, collect, write, take) = EAGER")
print("• Transformations build an execution plan")
print("• Actions trigger the actual computation")

TRANSFORMATIONS (Lazy Evaluation
Transformations don't execute immediately - they build a execution plan

Applying transformations...
Transformations applied (no computation happened yet!)
No data was actually processed - Spark just recorded the plan

ACTIONS (Eager Evaluation)
Actions trigger actual computation

Action 1: show() - This EXECUTES the transformations
+-----------+-------------+---------+
|fare_amount|trip_distance|high_fare|
+-----------+-------------+---------+
|       13.0|          3.7|    false|
|       41.0|         14.1|    false|
|       27.0|          9.6|    false|
|       17.0|         5.65|    false|
|       10.5|         2.63|    false|
+-----------+-------------+---------+
only showing top 5 rows
Execution time: 0.49 seconds

Action 2: Display aggregation - Another action that triggers execution
+--------+
|   total|
+--------+
|37557278|
+--------+

Execution time: 21.14 seconds

Action 3: collect() equivalent - Collecting sample data
Collected 10 rows
Exec

### Write processed results as Delta tables

In [0]:
print("Writing Results as Delta Tables\n")

# Write 1: Borough analysis
print("1. Writing borough analysis...")
borough_analysis.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("borough_analysis")
print("Borough analysis saved")

# Write 2: Hourly analysis
print("\n2. Writing hourly analysis...")
hourly_analysis.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("hourly_analysis")
print("Hourly analysis saved")

# Write 3: Sample of processed trips (limit to avoid long execution)
print("\n3. Writing sample processed trips...")
trips_with_zones.limit(100000) \
    .write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("processed_trips_sample")
print("Processed trips sample saved")

print("\n All tables saved successfully as Delta format (Parquet files)")

Writing Results as Delta Tables

1. Writing borough analysis...
Borough analysis saved

2. Writing hourly analysis...
Hourly analysis saved

3. Writing sample processed trips...
Processed trips sample saved

 All tables saved successfully as Delta format (Parquet files)


### Show execution plans for performance analysis

In [0]:
print("EXECUTION PLAN 1: Borough Analysis with Join\n")
print("This shows filter pushdown, join strategy, and aggregation execution\n")
borough_analysis.explain(mode="formatted")

print("\n" + "="*80 + "\n")

print("EXECUTION PLAN 2: SQL Query (Top Zones)\n")
top_zones_revenue.explain(mode="formatted")

print("\n" + "="*80 + "\n")

print("EXECUTION PLAN 3: Complex Query with Multiple Transformations\n")
trips_with_zones.filter(col("pickup_borough") == "Manhattan") \
    .groupBy("time_of_day") \
    .agg(avg("fare_amount").alias("avg_fare")) \
    .explain(mode="formatted")

EXECUTION PLAN 1: Borough Analysis with Join

This shows filter pushdown, join strategy, and aggregation execution

== Physical Plan ==
AdaptiveSparkPlan (25)
+- == Initial Plan ==
   ColumnarToRow (24)
   +- PhotonResultStage (23)
      +- PhotonSort (22)
         +- PhotonShuffleExchangeSource (21)
            +- PhotonShuffleMapStage (20)
               +- PhotonShuffleExchangeSink (19)
                  +- PhotonGroupingAgg (18)
                     +- PhotonShuffleExchangeSource (17)
                        +- PhotonShuffleMapStage (16)
                           +- PhotonShuffleExchangeSink (15)
                              +- PhotonGroupingAgg (14)
                                 +- PhotonProject (13)
                                    +- PhotonBroadcastHashJoin Inner (12)
                                       :- PhotonProject (5)
                                       :  +- PhotonProject (4)
                                       :     +- PhotonFilter (3)
                  

### Final results

In [0]:
print("FINAL PIPELINE RESULTS\n")

print("1. Borough Analysis Summary:")
spark.table("borough_analysis").orderBy(col("total_trips").desc()).show(15, truncate=False)

print("\n2. Top Revenue Generating Zones:")
top_zones_revenue.show(10, truncate=False)

print("\n3. Hourly Traffic Patterns:")
spark.table("hourly_analysis").orderBy("pickup_hour", "is_weekend").show(20, truncate=False)

print("\nPipeline completed successfully")
print("Data processed: ~84 million taxi trips")
print("Results written to Delta tables")

FINAL PIPELINE RESULTS

1. Borough Analysis Summary:
+--------------+-----------+-----------+------------------+------------------+------------------+--------------------+
|pickup_borough|time_of_day|total_trips|avg_distance      |avg_fare          |avg_tip_pct       |total_revenue       |
+--------------+-----------+-----------+------------------+------------------+------------------+--------------------+
|Manhattan     |Afternoon  |24482292   |2.259333383900441 |11.676600011551207|17.941164545769745|4.182052595740761E8 |
|Manhattan     |Evening    |21923754   |2.2943607025511636|10.96976241979362 |19.965456990615134|3.668507756150324E8 |
|Manhattan     |Morning    |18392531   |2.2634620096603126|11.151978900021971|18.45248935449638 |2.991516391039359E8 |
|Manhattan     |Night      |9714469    |2.9589932182603356|11.941787978323868|19.07010900338789 |1.7150225643108362E8|
|Queens        |Afternoon  |1903266    |11.82270426729642 |38.4647988878065  |16.465685327225646|9.500260270988744